# Ungraded Lab: Knowledge Distillation with Titanic Dataset

In this lab you will perform **knowledge distillation** where a small `student` model learns from a more complex `teacher` model. You will:
1. Define a `Distiller` class with custom distillation logic.
2. Train a large `teacher` RandomForestClassifier.
3. Train a `student` DecisionTreeClassifier using knowledge distillation.
4. Train another student from scratch without distillation.
5. Compare the three models.

## Imports

In [ ]:
import os
os.environ['PYTHONHASHSEED'] = str(42)
import random, numpy as np, pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
random.seed(42)
np.random.seed(42)

## Prepare the Data

Use the Titanic dataset — binary classification (survived = 0 or 1).

In [ ]:
def load_titanic():
    df = sns.load_dataset('titanic')
    df.drop(['class','who','adult_male','deck','embark_town','alive','alone'], axis=1, inplace=True)
    df['age'].fillna(df['age'].median(), inplace=True)
    df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)
    df['sex'] = df['sex'].map({'male':1,'female':0})
    df['embarked'] = LabelEncoder().fit_transform(df['embarked'])
    return df

df = load_titanic()
X = df.drop('survived', axis=1).values
y = df['survived'].values

splits = [0.7, 0.2, 0.1]
n = len(X)
i1, i2 = int(n*0.7), int(n*0.9)
idx = np.random.permutation(n)
X_train, y_train = X[idx[:i1]], y[idx[:i1]]
X_val,   y_val   = X[idx[i1:i2]], y[idx[i1:i2]]
X_test,  y_test  = X[idx[i2:]], y[idx[i2:]]

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## Code the Custom `Distiller` Class

Create a custom `Distiller` that uses the teacher's soft probability outputs to guide the student's training — analogous to KL divergence in neural network distillation.

In [ ]:
class Distiller:
    '''
    Knowledge Distillation from a RandomForest teacher to a DecisionTree student.

    The teacher's predicted probabilities (soft labels) are blended with hard labels
    to provide richer training signal to the student.
    '''

    def __init__(self, student, teacher):
        self.teacher = teacher
        self.student = student

    def compile(self, alpha=0.05, temperature=1.0):
        '''alpha: weight for hard labels (1-alpha: weight for soft labels)'''
        self.alpha = alpha
        self.temperature = temperature

    def fit(self, X_train, y_train):
        '''Train the student using teacher's soft predictions.'''
        # Teacher forward pass (not retrained)
        teacher_probs = self.teacher.predict_proba(X_train)[:, 1]

        # Blend hard labels with teacher soft labels
        soft_labels = self.alpha * y_train + (1 - self.alpha) * teacher_probs

        # Threshold to binary labels for the student
        blended_labels = (soft_labels > 0.5).astype(int)

        # Student learns from blended labels
        self.student.fit(X_train, blended_labels)

    def predict(self, X):
        return self.student.predict(X)

    def evaluate(self, X, y):
        y_pred = self.predict(X)
        return {'accuracy': accuracy_score(y, y_pred)}

## Teacher and Student Models

The teacher is a large RandomForest (complex, high accuracy). The student is a shallow DecisionTree (simple, fast).

In [ ]:
def create_big_model():
    '''Teacher: large RandomForestClassifier'''
    np.random.seed(42)
    return RandomForestClassifier(
        n_estimators=200, max_depth=None,
        min_samples_split=2, random_state=42
    )

def create_small_model():
    '''Student: shallow DecisionTreeClassifier'''
    np.random.seed(42)
    return DecisionTreeClassifier(
        max_depth=5, random_state=42
    )

In [ ]:
teacher = create_big_model()
student = create_small_model()
student_scratch = create_small_model()

def num_params(model):
    if hasattr(model, 'n_estimators'):
        return model.n_estimators * 2**(model.max_depth or 15) if model.max_depth else f'{model.n_estimators} trees (unlimited depth)'
    return f'max_depth={model.max_depth}'

print(f"Teacher model: {model.__class__.__name__} — {num_params(teacher)}")
print(f"Student model: {student.__class__.__name__} — {num_params(student)}")

### Train the Teacher

Knowledge distillation assumes the teacher is already trained.

In [ ]:
teacher.fit(X_train, y_train)
teacher_acc = accuracy_score(y_test, teacher.predict(X_test))
print(f"Teacher accuracy: {teacher_acc:.4f}")

## Train a Student from Scratch for Reference

Train a model equivalent to the student but without distillation.

In [ ]:
student_scratch.fit(X_train, y_train)
scratch_acc = accuracy_score(y_test, student_scratch.predict(X_test))
print(f"Student (scratch) accuracy: {scratch_acc:.4f}")

## Knowledge Distillation

Use the custom `Distiller` class to train the student with the teacher's guidance.

In [ ]:
distiller = Distiller(student=student, teacher=teacher)
distiller.compile(alpha=0.05, temperature=1.0)
distiller.fit(X_train, y_train)
distiller_acc = distiller.evaluate(X_test, y_test)['accuracy']
print(f"Distilled student accuracy: {distiller_acc:.4f}")

## Comparing the Models

In [ ]:
teacher_val_acc = accuracy_score(y_test, teacher.predict(X_test))
scratch_val_acc  = accuracy_score(y_test, student_scratch.predict(X_test))

print(f"Teacher:                         {teacher_val_acc*100:.2f}%")
print(f"Student with distillation:       {distiller_acc*100:.2f}%")
print(f"Student without distillation:    {scratch_val_acc*100:.2f}%")

In [ ]:
# Plot comparison
labels = ['Teacher\n(200 trees)', 'Student\n(distilled)', 'Student\n(scratch)']
accs   = [teacher_val_acc, distiller_acc, scratch_val_acc]
colors = ['#2196F3','#4CAF50','#FF9800']

fig, ax = plt.subplots(figsize=(8,5))
bars = ax.bar(labels, accs, color=colors, edgecolor='white', linewidth=1.5)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy')
ax.set_title('Knowledge Distillation — Titanic Dataset')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{acc:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

The teacher yields higher accuracy thanks to its ensemble of 200 trees. The **distilled student outperforms the student trained from scratch** — it learned the regularization and generalization patterns from the teacher.

**Congratulations on finishing this ungraded lab!**